<a href="https://colab.research.google.com/github/baekkii/randomlunch/blob/main/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OIBC 2026 3회차 — 기준선

하루전 기상예보로 발전소별 설비이용률을 LightGBM으로 예측하고 집합 단위로 합산합니다. 라운드 10·17 예측의 평균이 최종 입찰입니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 설정 / 라이브러리

In [2]:
from __future__ import annotations
import argparse, os, sys, time, subprocess
import numpy as np
import pandas as pd
import lightgbm as lgb

DATA_DIR = "/content/drive/MyDrive/2026_OIBC/01_participant_dataset"   # 데이터 폴더

TRAIN_FROM = pd.Timestamp("2023-06-01 01:00", tz="Asia/Seoul")   # 최근 2년만 사용
ROUNDS = (10, 17)
WA_COLS = ["wa_cloud", "wa_temp", "wa_humidity", "wa_rain",
           "wa_wind_speed", "wa_ground_press"]
WB_COLS = ["wb_cloud", "wb_temp", "wb_rel_hum", "wb_uv_idx", "wb_precip_prob",
           "wb_total_liq", "wb_vis", "wb_wind_speed", "wb_dew_point"]

t0 = time.time()
log = lambda m: print(f"[{time.time()-t0:6.1f}s] {m}", flush=True)


## 함수 정의

In [3]:
def solar(ts: pd.DatetimeIndex, lat: np.ndarray, lon: np.ndarray):
    """태양 고도·방위 (NOAA 근사, 도 단위). 외부 의존성 없이 계산."""
    doy = ts.dayofyear.to_numpy()
    hour = ts.hour.to_numpy() + ts.minute.to_numpy() / 60.0
    g = 2 * np.pi / 365.0 * (doy - 1 + (hour - 12) / 24.0)
    eqtime = 229.18 * (0.000075 + 0.001868*np.cos(g) - 0.032077*np.sin(g)
                       - 0.014615*np.cos(2*g) - 0.040849*np.sin(2*g))     # 분
    decl = (0.006918 - 0.399912*np.cos(g) + 0.070257*np.sin(g)
            - 0.006758*np.cos(2*g) + 0.000907*np.sin(2*g)
            - 0.002697*np.cos(3*g) + 0.00148*np.sin(3*g))                 # rad
    tst = (hour - 0.5) * 60 + eqtime + 4 * (lon - 135.0)
    ha = np.radians(tst / 4.0 - 180.0)
    la = np.radians(lat)
    cosz = np.sin(la)*np.sin(decl) + np.cos(la)*np.cos(decl)*np.cos(ha)
    cosz = np.clip(cosz, -1, 1)
    elev = 90.0 - np.degrees(np.arccos(cosz))
    az = np.degrees(np.arctan2(-np.sin(ha),
                               np.tan(decl)*np.cos(la) - np.sin(la)*np.cos(ha))) % 360
    return elev.astype("float32"), az.astype("float32"), cosz.astype("float32")


def features(gen, sites, fa, fb, index, r: int) -> pd.DataFrame:
    """(site, ts) 격자에 예보·태양기하를 붙인 특징 행렬."""
    if gen is not None:
        base = gen[["site_id", "ts", "energy"]]
    else:
        base = pd.MultiIndex.from_product([sites.site_id, index],
                                          names=["site_id", "ts"]).to_frame(index=False)
    d = base.merge(sites, on="site_id", how="left")
    d = d.merge(fa[fa["round"] == r].drop(columns="round"), on=["wa_station", "ts"], how="left")
    d = d.merge(fb[fb["round"] == r].drop(columns="round"), on=["wb_station", "ts"], how="left")
    ts = pd.DatetimeIndex(d.ts)
    d["elev"], d["azim"], d["cosz"] = solar(ts, d.lat.to_numpy(), d.lon.to_numpy())
    doy = ts.dayofyear.to_numpy()
    d["doy_sin"] = np.sin(2*np.pi*doy/365.25).astype("float32")
    d["doy_cos"] = np.cos(2*np.pi*doy/365.25).astype("float32")
    d["hour"] = ts.hour.to_numpy().astype("int8")
    d["cap"] = d.capacity_kw.astype("float32")
    return d


FEATS = WA_COLS + WB_COLS + ["elev", "azim", "cosz", "doy_sin", "doy_cos",
                              "hour", "cap", "lat", "lon"]


In [4]:
def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description="OIBC 2026 3회차 기준선 제출 생성 (Colab)")
    ap.add_argument("--data-dir", default=DATA_DIR, help="sites.csv, smp.csv, train/, test/ 가 있는 폴더")
    ap.add_argument("--out-dir", default=None, help="결과 저장 폴더 (기본: data-dir)")
    ap.add_argument("--sites-out", default=None,
                    help="발전소별 예측을 저장할 parquet 경로 (구성 최적화용)")
    args, _ = ap.parse_known_args(argv if argv is not None else sys.argv[1:])

    HERE = args.data_dir
    OUT = args.out_dir or HERE

    sites = pd.read_csv(f"{HERE}/sites.csv")[
        ["site_id", "current_group", "capacity_kw", "lat", "lon",
         "wa_station", "wb_station"]]
    scope = sites
    log(f"발전소 {len(sites)}개소 (전부 구성·예측 대상)")

    gen = pd.read_parquet(f"{HERE}/train/generation.parquet")
    gen["ts"] = pd.to_datetime(gen.ts)
    gen = gen[gen.ts >= TRAIN_FROM]
    log(f"학습 발전량 {len(gen):,}행 ({gen.ts.min():%Y-%m-%d} ~ {gen.ts.max():%Y-%m-%d})")

    fa_tr = pd.read_parquet(f"{HERE}/train/forecast_wa.parquet",
                            columns=["wa_station", "round", "ts"] + WA_COLS)
    fb_tr = pd.read_parquet(f"{HERE}/train/forecast_wb.parquet",
                            columns=["wb_station", "round", "ts"] + WB_COLS)
    for d in (fa_tr, fb_tr):
        d["ts"] = pd.to_datetime(d.ts)
    fa_tr = fa_tr[fa_tr.ts >= TRAIN_FROM]; fb_tr = fb_tr[fb_tr.ts >= TRAIN_FROM]
    log(f"학습 예보 A {len(fa_tr):,} / B {len(fb_tr):,}")

    models = {}
    for r in ROUNDS:
        d = features(gen, sites, fa_tr, fb_tr, None, r)
        d = d[d.elev > 0]
        y = (d.energy / d.cap).clip(0, 1.2).astype("float32")
        X = d[FEATS].astype("float32")
        log(f"round={r} 학습 행렬 {X.shape}")
        models[r] = lgb.LGBMRegressor(
            n_estimators=600, learning_rate=0.06, num_leaves=127,
            min_child_samples=200, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, verbose=-1, n_jobs=-1).fit(X, y)
        log(f"round={r} 학습 완료  train MAE {np.abs(models[r].predict(X)-y).mean():.4f}")
        del d, X, y

    del gen, fa_tr, fb_tr
    index = pd.DatetimeIndex(pd.to_datetime(pd.read_csv(f"{HERE}/smp.csv").ts)
                             .dt.tz_convert("Asia/Seoul"))
    log(f"채점 시각 {len(index):,}h ({index[0]} ~ {index[-1]})")

    fa_te = pd.read_parquet(f"{HERE}/test/forecast_wa.parquet",
                            columns=["wa_station", "round", "ts"] + WA_COLS)
    fb_te = pd.read_parquet(f"{HERE}/test/forecast_wb.parquet",
                            columns=["wb_station", "round", "ts"] + WB_COLS)
    for d in (fa_te, fb_te):
        d["ts"] = pd.to_datetime(d.ts)

    grp = scope.set_index("site_id").current_group.map({"A": "G1", "B": "G2"})
    out, per_site = [], []
    for r in ROUNDS:
        d = features(None, scope, fa_te, fb_te, index, r)
        X = d[FEATS].astype("float32")
        p = np.where(d.elev.to_numpy() > 0, models[r].predict(X), 0.0)
        d["kwh"] = np.clip(p, 0, None) * d.cap.to_numpy()
        d["group"] = d.site_id.map(grp)
        out.append(d.groupby(["group", "ts"], as_index=False).kwh.sum())
        if args.sites_out:
            per_site.append(d[["site_id", "ts", "kwh"]])
        log(f"round={r} 예측 완료  집합 합계 {d.kwh.sum()/1e6:,.2f} GWh")

    f = (pd.concat(out).groupby(["group", "ts"], as_index=False).kwh.mean()
           .rename(columns={"kwh": "forecast_kwh"})
           .sort_values(["group", "ts"]))
    f.to_csv(f"{OUT}/forecast.csv", index=False) # forecast.csv 저장
    grp.rename("group").reset_index().to_csv(f"{OUT}/composition.csv", index=False) # composition.csv 저장
    log(f"저장: {OUT}/composition.csv ({len(grp)}행) / forecast.csv ({len(f):,}행)")
    if args.sites_out:
        ps = pd.concat(per_site).groupby(["site_id", "ts"], as_index=False).kwh.mean()
        ps.to_parquet(args.sites_out, index=False, compression="zstd")
        log(f"저장: {args.sites_out} ({len(ps):,}행)")


## 실행
`--sites-out` 을 추가하면 발전소별 예측 parquet 도 저장

In [5]:
main(["--data-dir", DATA_DIR, "--sites-out", f"{DATA_DIR}/sites_pred.parquet"])

[  14.2s] 발전소 253개소 (전부 구성·예측 대상)
[  17.0s] 학습 발전량 3,427,240행 (2023-06-01 ~ 2025-06-01)
[  30.4s] 학습 예보 A 2,140,304 / B 2,559,745
[  36.6s] round=10 학습 행렬 (1710958, 24)
[ 217.9s] round=10 학습 완료  train MAE 0.0472
[ 224.0s] round=17 학습 행렬 (1710958, 24)
[ 404.5s] round=17 학습 완료  train MAE 0.0469
[ 405.4s] 채점 시각 8,760h (2025-06-01 01:00:00+09:00 ~ 2026-06-01 00:00:00+09:00)
[ 500.8s] round=10 예측 완료  집합 합계 59.13 GWh
[ 595.8s] round=17 예측 완료  집합 합계 59.00 GWh
[ 596.0s] 저장: /content/drive/MyDrive/2026_OIBC/01_participant_dataset/composition.csv (253행) / forecast.csv (17,520행)
